# Marching Cubes Transpose Invariance

This notebook tests whether `skimage.measure.marching_cubes` is transpose invariant for 3D inputs.

Fully auto-generated by Gemini.

In [1]:
import numpy as np
import skimage.measure as skm
import transpose_invariance as tpi
from skimage.util import img_as_float

In [2]:
imgs = tpi.get_3d_images()
# Use nuclei data, which has complex blobs
img = img_as_float(imgs[1][:20, :64, :64])
# Add a tiny bit of noise to create ambiguous cases (saddles)
rng = np.random.default_rng(42)
img += rng.normal(0, 0.01, size=img.shape)

In [3]:
def sort_mesh(verts, faces):
    # Sort vertices and get mapping
    # We round to avoid floating point issues during sorting
    verts_rounded = np.round(verts, 6)
    sort_idx = np.lexsort(verts_rounded.T)
    sorted_verts = verts[sort_idx]
    
    # Create mapping from old index to new index
    inv_sort_idx = np.zeros_like(sort_idx)
    inv_sort_idx[sort_idx] = np.arange(len(sort_idx))
    
    # Update faces
    sorted_faces = inv_sort_idx[faces]
    # Sort each face internally
    sorted_faces.sort(axis=1)
    # Sort faces by first, then second, then third index
    sorted_faces = sorted_faces[np.lexsort(sorted_faces.T)]
    
    return sorted_verts, sorted_faces

In [4]:
def test_marching_cubes_invariance():
    level = 0.5 * (img.min() + img.max())
    axes = (2, 1, 0)
    
    # Original
    verts_orig, faces_orig, _, _ = skm.marching_cubes(img, level)
    
    # Transposed
    img_r = np.transpose(img, axes)
    verts_r, faces_r, _, _ = skm.marching_cubes(img_r, level)
    
    # Map vertices back
    spatial_back = np.argsort(axes)
    verts_rolled_back = verts_r[:, spatial_back]
    
    sv_orig, sf_orig = sort_mesh(verts_orig, faces_orig)
    sv_rolled, sf_rolled = sort_mesh(verts_rolled_back, faces_r)
    
    print(f"Num vertices: {len(sv_orig)} vs {len(sv_rolled)}")
    print(f"Num faces: {len(sf_orig)} vs {len(sf_rolled)}")
    
    if len(sv_orig) != len(sv_rolled):
        print("INVARIANCE FAILED: Different number of vertices!")
    elif len(sf_orig) != len(sf_rolled):
        print("INVARIANCE FAILED: Different number of faces!")
    else:
        v_diff = np.abs(sv_orig - sv_rolled).max()
        f_diff = np.abs(sf_orig - sf_rolled).max()
        print(f"Max vertex difference: {v_diff}")
        print(f"Max face index difference: {f_diff}")
        if f_diff > 0:
            print("INVARIANCE FAILED: Mesh topology (faces) differs!")

In [5]:
test_marching_cubes_invariance()

Num vertices: 1843 vs 1843
Num faces: 2778 vs 2778
Max vertex difference: 0.0
Max face index difference: 147
INVARIANCE FAILED: Mesh topology (faces) differs!


## Why is it not invariant?

The `marching_cubes` algorithm (specifically the Lewiner implementation) is **not transpose invariant** in the presence of topological ambiguities.

While the algorithm correctly interpolates vertex positions along edges (resulting in a `Max vertex difference` of `0.0`), it must make decisions about how to connect those vertices into triangles within a voxel. 

In "ambiguous" voxels (e.g., where face or internal saddles exist), the Lewiner algorithm uses asymptotic deciders and lookup tables that are sensitive to the relative orientation of the voxel's axes and the order of traversal. Even though the same number of vertices and faces might be generated, the **connectivity (which vertices form which triangles)** can change if the image is transposed. 

This is evidenced by the `Max face index difference` being non-zero even after sorting vertices and faces: the set of triangles produced is topologically different.